In [1]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma

Wikipedia Retriever

In [2]:
from langchain_community.retrievers import WikipediaRetriever

In [3]:
retriever = WikipediaRetriever(top_k_results=2, lang="en")

In [4]:
query = "the geopolitical history of india and pakistan from the perspective of a chinese"

# Get relevant Wikipedia documents
docs = retriever.invoke(query)

In [5]:
for i, doc in enumerate(docs):
    print(f"\n---Result---")
    print(f"Content:\n{doc.page_content}...")


---Result---
Content:
The India–Pakistan war of 1971, also known as the third Indo-Pakistani war, was a military confrontation between India and Pakistan that occurred during the Bangladesh Liberation War in East Pakistan from 3 December 1971 until the Pakistani capitulation in Dhaka on 16 December 1971.  The war began with Pakistan's Operation Chengiz Khan, consisting of preemptive aerial strikes on eight Indian air stations. The strikes led to India declaring war on Pakistan, marking their entry into the war for East Pakistan's independence, on the side of Bengali nationalist forces. India's entry expanded the existing conflict with Indian and Pakistani forces engaging on both the eastern and western fronts.
Thirteen days after the war started, India achieved a clear upper hand, and the Eastern Command of the Pakistan military signed the instrument of surrender on 16 December 1971 in Dhaka, marking the formation of East Pakistan as the new nation of Bangladesh. Approximately 93,000 

Vector Store Retriever

In [6]:
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document

In [7]:
documents = [
    Document(page_content="LangChain helps developers build LLM applications easily."),
    Document(page_content="Chroma is a vector database optimized for LLM-based search."),
    Document(page_content="Embeddings convert text into high-dimensional vectors."),
    Document(page_content="OpenAI provides powerful embedding models."),
]

In [10]:
vector_store = Chroma.from_documents(
    embedding=GoogleGenerativeAIEmbeddings(model="models/text-embedding-004"),
    persist_directory="my_chroma_db",
    collection_name="sample",
    documents=documents
)

In [12]:
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

In [13]:
query = "What is Chroma used for?"
result = retriever.invoke(query)

In [14]:
for i, doc in enumerate(result):
    print(f"\n---Result {i+1} ---")
    print(doc.page_content)


---Result 1 ---
Chroma is a vector database optimized for LLM-based search.

---Result 2 ---
Embeddings convert text into high-dimensional vectors.


Doubt - Vectorstore also has capability to do this, using vectorstore.similarity_search, so why do we need retriever? - Vectorstore can only search using similarity seach, if we need other methods we need to use retriever

MMR (Maximum Marginal Relevance)

Normal retriever may give results which are similar to each other, but MMR try to generate results which are not only relevent to query but which are different from each other

In [15]:
# Sample documents
docs = [
    Document(page_content="LangChain makes it easy to work with LLMs."),
    Document(page_content="LangChain is used to build LLM based applications."),
    Document(page_content="Chroma is used to store and search document embeddings."),
    Document(page_content="Embeddings are vector representations of text."),
    Document(page_content="MMR helps you get diverse results when doing similarity search."),
    Document(page_content="LangChain supports Chroma, FAISS, Pinecone, and more."),
]

In [16]:
vector_store2 = Chroma.from_documents(
    embedding=GoogleGenerativeAIEmbeddings(model="models/text-embedding-004"),
    persist_directory="my_chroma_db",
    collection_name="sample_MMR",
    documents=documents
)

In [23]:
retriever2 = vector_store2.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3, "lambda_mult": 0.5}
)

In [24]:
vector_store2._collection

Collection(name=sample_MMR)

In [25]:
query = "Wnat is Langachain?"
results = retriever2.invoke(query)

In [26]:
for i, doc in enumerate(results):
    print(f"\n---Result {i+1} ---")
    print(doc.page_content)


---Result 1 ---
LangChain helps developers build LLM applications easily.

---Result 2 ---
Embeddings convert text into high-dimensional vectors.

---Result 3 ---
Chroma is a vector database optimized for LLM-based search.


Multi Query Retriever

Generate multiple questions from user query and then retrieve answers from each of them and remove duplicated

In [27]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_classic.retrievers import MultiQueryRetriever

In [28]:
# Relevant health & wellness documents
all_docs = [
    Document(page_content="Regular walking boosts heart health and can reduce symptoms of depression.", metadata={"source": "H1"}),
    Document(page_content="Consuming leafy greens and fruits helps detox the body and improve longevity.", metadata={"source": "H2"}),
    Document(page_content="Deep sleep is crucial for cellular repair and emotional regulation.", metadata={"source": "H3"}),
    Document(page_content="Mindfulness and controlled breathing lower cortisol and improve mental clarity.", metadata={"source": "H4"}),
    Document(page_content="Drinking sufficient water throughout the day helps maintain metabolism and energy.", metadata={"source": "H5"}),
    Document(page_content="The solar energy system in modern homes helps balance electricity demand.", metadata={"source": "I1"}),
    Document(page_content="Python balances readability with power, making it a popular system design language.", metadata={"source": "I2"}),
    Document(page_content="Photosynthesis enables plants to produce energy by converting sunlight.", metadata={"source": "I3"}),
    Document(page_content="The 2022 FIFA World Cup was held in Qatar and drew global energy and excitement.", metadata={"source": "I4"}),
    Document(page_content="Black holes bend spacetime and store immense gravitational energy.", metadata={"source": "I5"}),
]

In [29]:
model = ChatGoogleGenerativeAI(model = "gemini-2.5-flash")

In [45]:
vector_store3 = Chroma.from_documents(
    embedding=GoogleGenerativeAIEmbeddings(model="models/text-embedding-004"),
    persist_directory="my_chroma_db",
    collection_name="sample_MultiQuery",
    documents=all_docs
)

In [46]:
similarity_retriever = vector_store3.as_retriever(search_type="similarity", search_kwargs={"k":5})

In [47]:
multiquery_retriever = MultiQueryRetriever.from_llm(
    retriever=vector_store3.as_retriever(search_kwargs={"k":5}),
    llm = model
)

In [48]:
query_new = "How to improve enery levels and maintain balance?"

In [49]:
# Retrieves results
similarity_results = similarity_retriever.invoke(query_new)
multiquery_results = multiquery_retriever.invoke(query_new)

In [53]:
print("Similairty Results - ")
for i, doc in enumerate(similarity_results):
    print(f"\n---Result {i+1} ---")
    print(doc.page_content)
 
print("\n"+"*"*150)
   
print("\nMultiQuery Results -")
for i, doc in enumerate(multiquery_results):
    print(f"\n---Result {i+1} ---")
    print(doc.page_content)


Similairty Results - 

---Result 1 ---
Drinking sufficient water throughout the day helps maintain metabolism and energy.

---Result 2 ---
Mindfulness and controlled breathing lower cortisol and improve mental clarity.

---Result 3 ---
Regular walking boosts heart health and can reduce symptoms of depression.

---Result 4 ---
Consuming leafy greens and fruits helps detox the body and improve longevity.

---Result 5 ---
The solar energy system in modern homes helps balance electricity demand.

******************************************************************************************************************************************************

MultiQuery Results -

---Result 1 ---
Drinking sufficient water throughout the day helps maintain metabolism and energy.

---Result 2 ---
Mindfulness and controlled breathing lower cortisol and improve mental clarity.

---Result 3 ---
Consuming leafy greens and fruits helps detox the body and improve longevity.

---Result 4 ---
Deep sleep is crucia

Contextual Compression Retriever

The Contextual Compression Retriever compress documents after retrieval — keeping only the relevant content based on user query

In [56]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor

In [55]:
# Recreate the document objects from the previous data
docs = [
    Document(page_content=(
        """The Grand Canyon is one of the most visited natural wonders in the world.
        Photosynthesis is the process by which green plants convert sunlight into energy.
        Millions of tourists travel to see it every year. The rocks date back millions of years."""
    ), metadata={"source": "Doc1"}),

    Document(page_content=(
        """In medieval Europe, castles were built primarily for defense.
        The chlorophyll in plant cells captures sunlight during photosynthesis.
        Knights wore armor made of metal. Siege weapons were often used to breach castle walls."""
    ), metadata={"source": "Doc2"}),

    Document(page_content=(
        """Basketball was invented by Dr. James Naismith in the late 19th century.
        It was originally played with a soccer ball and peach baskets. NBA is now a global league."""
    ), metadata={"source": "Doc3"}),

    Document(page_content=(
        """The history of cinema began in the late 1800s. Silent films were the earliest form.
        Thomas Edison was among the pioneers. Photosynthesis does not occur in animal cells.
        Modern filmmaking involves complex CGI and sound design."""
    ), metadata={"source": "Doc4"})
]

In [59]:
vector_store4 = Chroma.from_documents(
    embedding=GoogleGenerativeAIEmbeddings(model="models/text-embedding-004"),
    persist_directory="my_chroma_db",
    collection_name="sample_ContextualCompression",
    documents=docs
)

In [60]:
base_retriever = vector_store4.as_retriever(search_kwargs={"k":5})

In [57]:
model = ChatGoogleGenerativeAI(model = "gemini-2.5-flash")
compressor = LLMChainExtractor.from_llm(model)

In [61]:
# Create the contextual compression retriever
compression_retriever = ContextualCompressionRetriever(
    base_retriever=base_retriever,
    base_compressor=compressor
)

In [62]:
# Query the retriever
query = "What is photosynthesis?"
compressed_results = compression_retriever.invoke(query)

In [63]:
for i, doc in enumerate(compressed_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)



--- Result 1 ---
Photosynthesis is the process by which green plants convert sunlight into energy.

--- Result 2 ---
The chlorophyll in plant cells captures sunlight during photosynthesis.
